In [9]:
import scanpy as sc #for seq data
import squidpy as sq #for spatial data
import scvi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

scvi.settings.seed = 42
print(f"scvi-tools version: {scvi.__version__}")

Seed set to 42


scvi-tools version: 1.3.3


In [ ]:
# Load datasets, remember datasets are already normalised and log transformed
adata_sp = sq.datasets.visium_hne_adata()
adata_sc = sq.datasets.sc_mouse_cortex()

print(f"Spatial:   {adata_sp.shape}")
print(f"scRNA-seq: {adata_sc.shape}")

# Check where raw counts might be stored
print(f"\nadata_sc.raw: {adata_sc.raw}")
print(f"adata_sc.layers: {list(adata_sc.layers.keys())}")
print(f"adata_sp.raw: {adata_sp.raw}")
print(f"adata_sp.layers: {list(adata_sp.layers.keys())}")

Spatial:   (2688, 18078)
scRNA-seq: (21697, 36826)

adata_sc.raw: Raw AnnData with n_obs × n_vars = 21697 × 36826
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells'
adata_sc.layers: []
adata_sp.raw: Raw AnnData with n_obs × n_vars = 2688 × 18078
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells'
adata_sp.layers: []


In [ ]:
import scipy.sparse as sp

# gimVI requires RAW counts — stored in .raw for both datasets
adata_sc_raw = adata_sc.raw.to_adata()
adata_sp_raw = adata_sp.raw.to_adata()

# Filter extreme cells (very high library sizes cause numerical overflow in ZINB)
sc.pp.filter_cells(adata_sc_raw, min_genes=200)
sc.pp.filter_cells(adata_sc_raw, max_genes=5000)
sc.pp.filter_genes(adata_sc_raw, min_cells=10)

sc.pp.filter_cells(adata_sp_raw, min_genes=200)
sc.pp.filter_genes(adata_sp_raw, min_cells=10)

print(f"scRNA-seq after filtering: {adata_sc_raw.shape}")
print(f"Spatial after filtering:   {adata_sp_raw.shape}")

# Find shared genes
shared_genes = list(set(adata_sc_raw.var_names) & set(adata_sp_raw.var_names))
print(f"Shared genes: {len(shared_genes)}")

# Subset to shared genes
adata_sc_shared = adata_sc_raw[:, shared_genes].copy()
adata_sp_shared = adata_sp_raw[:, shared_genes].copy()

# Copy cell type labels (reindex to match filtered cells)
adata_sc_shared.obs = adata_sc.obs.loc[adata_sc_shared.obs_names].copy()

adata_sc_shared.X = adata_sc_shared.X.astype(np.float32)
adata_sp_shared.X = adata_sp_shared.X.astype(np.float32)

# Verify counts look reasonable
X = adata_sc_shared.X
if sp.issparse(X): X = X.toarray()
print(f"\nscRNA-seq counts - min: {X.min():.0f}, max: {X.max():.0f}")

In [ ]:
# Set up gimVI model
from scvi.external import GIMVI

GIMVI.setup_anndata(adata_sc_shared)
GIMVI.setup_anndata(adata_sp_shared)

model = GIMVI(adata_sc_shared, adata_sp_shared)
print(model)

In [ ]:
# Train the model with lower learning rate for numerical stability
model.train(max_epochs=200, plan_kwargs={'lr': 1e-4})

In [ ]:
# Get imputed gene expression for spatial spots
# gimVI imputes the full scRNA-seq gene set at each spatial spot
_, imputed = model.get_imputed_values(normalized=True)
print(f"Imputed shape: {imputed.shape}")

In [ ]:
import anndata

# Build AnnData with imputed spatial expression
adata_imputed = anndata.AnnData(X=imputed)
adata_imputed.obs_names = adata_sp_shared.obs_names
adata_imputed.var_names = adata_sp_shared.var_names
adata_imputed.obs = adata_sp_shared.obs.copy()
adata_imputed.obsm['spatial'] = adata_sp_shared.obsm['spatial']

print(adata_imputed)

In [ ]:
# Visualize imputed expression of Lamp5 - known cortical layer marker
gene = 'Lamp5' if 'Lamp5' in adata_imputed.var_names else adata_imputed.var_names[0]
print(f"Visualising: {gene}")

sc.pl.spatial(adata_imputed, color=gene, spot_size=150, title=f'gimVI Imputed: {gene}')

In [ ]:
# Get latent representation - the shared embedding of both modalities
latent_sc, latent_sp = model.get_latent_representation()
print(f"scRNA-seq latent: {latent_sc.shape}")
print(f"Spatial latent:   {latent_sp.shape}")

# Add to AnnData for visualization
adata_sc_shared.obsm['X_gimVI'] = latent_sc
adata_sp_shared.obsm['X_gimVI'] = latent_sp

# UMAP of the shared latent space
import scvi
sc.pp.neighbors(adata_sc_shared, use_rep='X_gimVI')
sc.tl.umap(adata_sc_shared)
sc.pl.umap(adata_sc_shared, color='cell_subclass', title='scRNA-seq in gimVI latent space')